# Lesson 11 — Growth-optimal is not riskless, and the difference is the trade

Sizing a family of mutually exclusive contracts is not the scalar Kelly formula repeated. Exactly one outcome pays, so a dollar on one is partly a hedge for the dollar on another, and the log-optimal split has an exact solution over the joint distribution.

**The rule.** `fₛ = qₛ − R·aₛ,  R = (1 − Σq) / (1 − Σa)`

**When it holds.** Over one mutually exclusive family, priced at what each outcome costs to buy, with the whole family present.

**When it fails.** Mistaking the Kelly plan for the arbitrage. Where a basket costs under a dollar both exist, and they are different portfolios: the Dutch book buys equal contracts and its profit is certain, while Kelly stakes the measure, grows faster and can lose a third of the bankroll on one settlement.

| | |
|---|---|
| Lesson id | `kelly` |
| Pane it appears on | `lattice` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/kelly.py` |
| Tests that go red if it stops being true | `tests/test_coherence_kelly.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. The textbook case, so the machinery can be checked by hand

In [ ]:
from modules.coherence.kernel import kelly

# The textbook case, so the machinery can be checked against a number a reader
# already knows: an even-money bet at 60/40. Kelly says stake a fifth.
textbook = kelly.solve(
    [
        kelly.Candidate("HEADS", "Heads", Decimal("0.60"), Decimal("0.50")),
        kelly.Candidate("TAILS", "Tails", Decimal("0.40"), Decimal("0.50")),
    ],
    shrinkage=Decimal(1),
)
for stake in textbook.stakes:
    print(f"  {stake.label:<8} q {stake.probability}  price {stake.price}  full fraction {stake.full_fraction}")
print()
print(f"  full fraction on the favourite : {textbook.stakes[0].full_fraction}")
print(f"  exactly one fifth              : {textbook.stakes[0].full_fraction == Decimal('0.20')}")
print(f"  cash held                      : {textbook.full_cash_fraction}")
print(f"  cash rate R                    : {textbook.reserve_rate}")
print()
print("  This is f* = (bp - q) / b at b = 1: (0.60 - 0.40) / 1 = 0.20. The exact solution")
print("  here reproduces it, because the scalar formula is the one-outcome case of the")
print("  same joint problem — and nothing but the joint problem is right past two outcomes.")

## 2. The case that matters: a family whose offers total under a dollar

In [ ]:
# The case that matters. Three mutually exclusive outcomes whose offers total
# under a dollar, so a riskless profit exists AND a growth-optimal plan exists,
# and they are not the same portfolio.
FAMILY = (
    ("A", "Outcome A", "0.5", "0.30"),
    ("B", "Outcome B", "0.3", "0.32"),
    ("C", "Outcome C", "0.2", "0.32"),
)
candidates = [
    kelly.Candidate(ticker, label, Decimal(probability), Decimal(price))
    for ticker, label, probability, price in FAMILY
]
full = kelly.solve(candidates, shrinkage=Decimal(1))

print(f"  {full.detail}")
print()
print("  outcome      q      price    q/price    full Kelly stake")
for stake in full.stakes:
    ratio = (stake.probability / stake.price).quantize(Decimal("0.0001"))
    print(f"  {stake.label:<11} {stake.probability}    {stake.price}     {ratio}     {stake.full_fraction}")
print()
print(f"  basket cost           {full.basket_cost}")
print(f"  arbitrage available   {full.arbitrage_available}")
print(f"  cash held             {full.cash_fraction}")
print(f"  cash rate R           {full.reserve_rate}   (zero: with the basket under a dollar there is no reason to hold cash)")

## 3. Growth-optimal is not riskless

In [ ]:
print(f"  riskless log growth   {full.riskless_growth.quantize(Decimal('0.000001'))}   = ln(1 / {full.basket_cost})")
print(f"  Kelly log growth      {full.growth_rate.quantize(Decimal('0.000001'))}")
print(f"  worst-case wealth     {full.worst_case_wealth.quantize(Decimal('0.0001'))} of the bankroll")
print()
print("  What each portfolio pays, per dollar of bankroll, in each state:")
print()
cash = Decimal(1) - sum((stake.full_fraction for stake in full.stakes), Decimal(0))
certain = Decimal(1) / full.basket_cost
print("  state        arbitrage basket   Kelly plan")
for stake in full.stakes:
    kelly_wealth = cash + stake.full_fraction / stake.price
    print(f"  {stake.label:<11}  {certain.quantize(Decimal('0.0001'))}           {kelly_wealth.quantize(Decimal('0.0001'))}")
print()
print("  THE KELLY PLAN IS NOT THE ARBITRAGE. The Dutch book buys an EQUAL NUMBER of")
print("  contracts of every outcome; that is what makes its payoff a flat dollar in every")
print("  state and its profit certain — the same 1.0638 in each row above. Kelly buys")
loss = (Decimal(1) - full.worst_case_wealth) * 100
print("  stakes in proportion to q, which is a different portfolio: it grows faster in the")
print(f"  long run and its worst state leaves {full.worst_case_wealth.quantize(Decimal('0.0001'))} of the bankroll — a {loss.quantize(Decimal('0.1'))}% loss")
print("  on a single settlement.")
print()
print("  A plan is therefore reported with its worst state next to its growth rate, and")
print("  where an arbitrage exists the certain alternative is priced beside it, so that")
print("  the two are never mistaken for each other.")

## 4. Why the shipped default is a quarter

In [ ]:
quarter = kelly.solve(candidates)
print(f"  shipped default shrinkage : {quarter.shrinkage}")
print(f"  full Kelly growth         : {quarter.full_growth_rate.quantize(Decimal('0.000001'))}")
print(f"  quarter Kelly growth      : {quarter.growth_rate.quantize(Decimal('0.000001'))}")
print(f"  full Kelly worst state    : {full.worst_case_wealth.quantize(Decimal('0.0001'))}")
print(f"  quarter Kelly worst state : {quarter.worst_case_wealth.quantize(Decimal('0.0001'))}")
print()
print("  Full Kelly maximises growth only if q is correct, and q here is inferred from")
print("  quotes that move. The growth curve is flat near the optimum and steep past it,")
print("  so over-betting costs far more than under-betting. Both fractions are reported")
print("  and the caller is told which one it is looking at.")

## 5. Where q comes from decides whether any of it means anything

In [ ]:
# Where the measure comes from decides whether any of this means anything.
NO_EDGE = (("A", "Outcome A", "0.35"), ("B", "Outcome B", "0.35"), ("C", "Outcome C", "0.35"))
mid_priced = kelly.solve([
    kelly.Candidate(ticker, label, Decimal(1) / 3, Decimal(price)) for ticker, label, price in NO_EDGE
])
print(f"  {mid_priced.detail}")
print(f"  cash held  {mid_priced.cash_fraction}")
print(f"  growth     {mid_priced.growth_rate}")
print()
print("  Feed this the market's own prices as the measure and it correctly tells you to")
print("  bet nothing, because you have no edge over the prices you are quoting back. The")
print("  measure worth feeding it is the COHERENT one — the nearest price vector that")
print("  admits a probability, which dutchbook and coherence_index already compute. Then")
print("  the edge being sized is the incoherence itself, which is the only edge this")
print("  engine ever claims to find.")